In [ ]:
from torchvision import datasets
from torchvision import transforms as T
from torch.utils.data import DataLoader, Subset
import torch
import torch.nn as nn
from torchinfo import summary
import matplotlib.pyplot as plt
import os
import torch.optim as optim
from tqdm import tqdm
from sklearn.model_selection import train_test_split

In [ ]:
USE_GRAYSCALE = False
BATCH_SIZE = 8
IMAGE_SIZE = 96 
IN_CHANNELS = 3 
NUM_CLASSES = 1
ALPHA = 0.25
NUM_EPOCHS = 100
LEARNING_RATE = 2e-3
L2_VALUE = 4e-6

In [ ]:
if USE_GRAYSCALE:
    transform = T.Compose([
        # ImageFolder will treat images as RGB images, even if they're stored as grayscale.
        T.Grayscale(),
        T.ToTensor(),
    ])
else:
    transform = T.Compose([
        T.ToTensor(),
    ])

data_dir = "../data/subset"
dataset = datasets.ImageFolder(data_dir, transform=transform)

In [ ]:

train_idx, val_idx = train_test_split(
    range(len(dataset)),
    test_size=0.2,
    stratify=dataset.targets,
    random_state=123
)

train_subset = Subset(dataset, train_idx)
val_subset = Subset(dataset, val_idx)

train_loader = DataLoader(train_subset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=8, shuffle=True, num_workers=2)

In [ ]:
print(dataset.classes, dataset.class_to_idx)

In [ ]:
for imgs, labels in train_loader:
    print(f"input shape: {imgs.shape}, label shape: {labels.shape}")
    print(f"input dtype: {imgs[0].dtype}, label dtype: {labels[0].dtype}")
    print(f"label sample: {labels}")
    break

In [ ]:
class FirstConv(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(in_channels=3,
                              out_channels=int(32*ALPHA),
                              kernel_size=3,
                              stride=2,
                              padding=1,
                              bias=False)
        self.bn = nn.BatchNorm2d(num_features=int(32*ALPHA))
        self.relu = nn.ReLU6()

    def forward(self, x):
        x = self.relu(self.bn(self.conv(x)))
        return x

In [ ]:
conv = FirstConv()
temp = torch.rand(1, 3, 224, 224)
x = conv(temp)
x.shape

In [ ]:
class DepthwiseSeparableConv(nn.Module):
        def __init__(self, in_channels, out_channels, downsample=False):
                super().__init__()

                in_channels = int(in_channels * ALPHA)
                out_channels = int(out_channels * ALPHA)
                stride = 2 if downsample else 1
                
                self.dwconv = nn.Conv2d(in_channels=in_channels,
                                        out_channels=in_channels,
                                        kernel_size=3,
                                        stride=stride,
                                        padding=1,
                                        groups=in_channels,
                                        bias=False)
                self.bn0 = nn.BatchNorm2d(num_features=in_channels)

                self.pwconv = nn.Conv2d(in_channels=in_channels,
                                        out_channels=out_channels,
                                        kernel_size=1,
                                        stride=1,
                                        padding=0,
                                        groups=1,
                                        bias=False
                                        )
                self.bn1 = nn.BatchNorm2d(num_features=out_channels)
                
                self.relu = nn.ReLU6()
            
        def forward(self, x):
                x = self.relu(self.bn0(self.dwconv(x)))
                x = self.relu(self.bn1(self.pwconv(x)))
                return x

In [ ]:
depthwise_sep_conv = DepthwiseSeparableConv(32, 64, False)

x = torch.randn(1, int(32*ALPHA), 112, 112)
x = depthwise_sep_conv(x)
x.shape

In [ ]:
depthwise_sep_conv = DepthwiseSeparableConv(64, 128, True)

x = depthwise_sep_conv(x)
x.shape

In [ ]:
class MobileNetV1(nn.Module):
    def __init__(self):
        super().__init__()

        self.first_conv = FirstConv()
        self.depthwise_sep_conv0 = DepthwiseSeparableConv(32, 64)
        self.depthwise_sep_conv1 = DepthwiseSeparableConv(64, 128, downsample=True)
        self.depthwise_sep_conv2 = DepthwiseSeparableConv(128, 128)
        self.depthwise_sep_conv3 = DepthwiseSeparableConv(128, 256, downsample=True)
        self.depthwise_sep_conv4 = DepthwiseSeparableConv(256, 256)
        self.depthwise_sep_conv5 = DepthwiseSeparableConv(256, 512, downsample=True)
        self.depthwise_sep_conv6 = nn.ModuleList(
            [DepthwiseSeparableConv(512, 512) for _ in range(5)]
        )
        self.depthwise_sep_conv7 = DepthwiseSeparableConv(512, 1024, downsample=True)
        self.depthwise_sep_conv8 = DepthwiseSeparableConv(1024, 1024)

        num_output_channels = self.depthwise_sep_conv8.pwconv.out_channels

        self.avgpool = nn.AdaptiveAvgPool2d(output_size=(1, 1))
        self.fc = nn.Linear(num_output_channels, NUM_CLASSES)
                                  
    def forward(self, x):
        x = self.first_conv(x)
        
        x = self.depthwise_sep_conv0(x)
        x = self.depthwise_sep_conv1(x)
        x = self.depthwise_sep_conv2(x)
        x = self.depthwise_sep_conv3(x)
        x = self.depthwise_sep_conv4(x)
        x = self.depthwise_sep_conv5(x)
        for layer in self.depthwise_sep_conv6:
            x = layer(x)
        x = self.depthwise_sep_conv7(x)
        x = self.depthwise_sep_conv8(x)
        
        x = self.avgpool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.fc(x)
        return x

In [ ]:
model = MobileNetV1()
x = torch.randn((BATCH_SIZE, IN_CHANNELS, IMAGE_SIZE, IMAGE_SIZE))
model(x)

In [ ]:
model = MobileNetV1()
summary(model, input_size=(1, 3, 96, 96))

In [ ]:
def training(model, train_loader, val_loader, num_epochs, learning_rate, l2_value, threshold):
    
    loss_fn = nn.BCEWithLogitsLoss()

    # Values are adapted from Wake Vision paper.
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=l2_value)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    train_losses =  []
    val_losses = []
    val_accuracies, val_precisions, val_recalls, val_f1s = [], [], [], []
    least_val_loss = float("inf")
    os.makedirs("../models", exist_ok=True)

    for epoch in range(num_epochs):

        model.train()
        train_loss = 0.0
        tp = fp = tn = fn = 0


        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}", leave=False)
        for imgs, y_actual in progress_bar:
            y_pred = model(imgs)
            loss = loss_fn(y_pred.squeeze(1), y_actual.float())
            probs = torch.sigmoid(y_pred.squeeze(1))
            preds = (probs >= threshold).long()
            y_actual = y_actual.long()

            tp += ((preds == 1) & (y_actual == 1)).sum().item()
            fp += ((preds == 1) & (y_actual == 0)).sum().item()
            tn += ((preds == 0) & (y_actual == 0)).sum().item()
            fn += ((preds == 0) & (y_actual == 1)).sum().item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        scheduler.step()
        train_acc = (tp+tn) / (tp+fp+tn+fn)
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        tqdm.write(f"Epoch {epoch+1}\nTrain -> Loss: {avg_train_loss:.4f}, TrainAcc: {train_acc:.4f}")

        model.eval()
        val_loss = 0.0
        tp = fp = tn = fn = 0
        with torch.no_grad():
            for imgs, y_actual in val_loader:
                y_pred = model(imgs)
                loss = loss_fn(y_pred.squeeze(1), y_actual.float())
                probs = torch.sigmoid(y_pred.squeeze(1))
                preds = (probs >= threshold).long()
                y_actual = y_actual.long()

                tp += ((preds == 1) & (y_actual == 1)).sum().item()
                fp += ((preds == 1) & (y_actual == 0)).sum().item()
                tn += ((preds == 0) & (y_actual == 0)).sum().item()
                fn += ((preds == 0) & (y_actual == 1)).sum().item()
                val_loss += loss.item()

            
            accuracy = (tp+tn) / (tp+fp+tn+fn)
            precision = tp / (tp+fp) if (tp+fp) else 0.0
            recall = tp / (tp+fn) if (tp+fn) else 0.0
            f1 = 2*precision*recall / (precision+recall) if (precision+recall) else 0.0

            avg_loss = val_loss / len(val_loader)
            val_losses.append(avg_loss)
            val_accuracies.append(accuracy)
            val_precisions.append(precision)
            val_recalls.append(recall)
            val_f1s.append(f1)
            tqdm.write(f"Validation -> Loss: {avg_loss:.4f}, Acc: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}\n ------------------")

            if(avg_loss < least_val_loss):
                least_val_loss = avg_loss
                torch.save(model.state_dict(), "../models/best_baseline_model.pth")
                tqdm.write("Best model saved.") 

    return model, train_losses, val_losses, val_accuracies, val_precisions, val_recalls, val_f1s 

In [ ]:
model = MobileNetV1()
trained_model, train_losses, val_losses, val_accuracies, val_precisions, val_recalls, val_f1s = training(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    l2_value=L2_VALUE,
    threshold=0.5)

**First full train/val run — overfitting confirmed early.** Best validation
loss (0.6666, F1 0.602) occurred at epoch 5 of 100; by epoch 100, train
accuracy reached 0.996 while validation loss rose to 1.978 and F1 fell to
0.585. The model stopped generalizing almost immediately and spent the
remaining epochs memorizing the training set. Confirms 480 training images
is far too little for this model capacity.

*(Note: no random seed is set, so re-running this cell will produce different
exact numbers each time — the overfitting pattern is consistent, the specific
epoch/loss values are not.)*

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.plot(epochs, train_losses, label="Train Loss")
ax1.plot(epochs, val_losses, label="Val Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("BinaryCrossEntropy Loss")
ax1.set_title("Loss")
ax1.legend()

ax2.plot(epochs, val_accuracies, label="Accuracy")
ax2.plot(epochs, val_precisions, label="Precision")
ax2.plot(epochs, val_recalls, label="Recall")
ax2.plot(epochs, val_f1s, label="F1")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Score")
ax2.set_ylim(0, 1)
ax2.set_title("Validation metrics")
ax2.legend()

plt.tight_layout()
plt.show()